In [ ]:
from matplotlib import pyplot as plt
from sklearn import datasets
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import math
# datasets = ['breast_cancer.csv', 'diabetes_prediction_dataset.csv']

## Check if there Null data

In [ ]:
for i in range(1, 106):
    df = pd.read_csv('ds'+ str(i) +'.csv')
    print('='*50)
    print('shape of {}:'.format(i), df.shape)
    print(df.columns)                            # features names and label
    if df.isnull().sum().sum() != 0:             # null?
        print(df.isnull().sum())
        break

## Definitions for Experiments

In [ ]:
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from matplotlib import pyplot as plt
from sklearn import metrics
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras import optimizers



################################ Predict ################################
def predict(result, X_test):
    predicted = []
    for j in range(X_test.shape[0]):
        if result[j] <= 0.5:
            predicted.append(0)
        else:
            predicted.append(1)
    return predicted

def predict_ours(result, X_test):
    predicted = []
    for j in range(X_test.shape[0]):
        if result[j] <= 0.5:
            predicted.append(0)
        else:
            predicted.append(1)
    return predicted

################################ MSE ################################
def MSE(y_true, y_pred):
    return tf.reduce_mean(tf.math.square(y_true - y_pred))

################################ BCE ################################
def BCE(y_true, y_pred):
    return -tf.reduce_mean(y_true*tf.math.log(y_pred)+(1-y_true)*tf.math.log(1-y_pred))

################################ Ours_Accu ################################
def Ours_Accu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    yl = y_train.shape[0]
    accu = (yl-tf.reduce_sum(y_true)-tf.reduce_sum(y_pred)+2*tf.reduce_sum(y_true*y_pred)) / yl
    return 1-accu

################################ Ours_Fbeta ################################
def Ours_Fbeta(y_true, y_pred):
#     beta = 1 
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    numerator = (1+beta**2)*tf.reduce_sum(y_true*y_pred)
    denominator = (beta**2)*tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return 1-(numerator/denominator)

################################ Ours_Gmean ################################
def Ours_Gmean(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    syhy = tf.reduce_sum(y_true*y_pred)
    sy = tf.reduce_sum(y_true)
    yl = y_train.shape[0]
#     gmean = syhy*(yl-tf.reduce_sum(y_pred)-sy+syhy)/(sy*(yl-sy))
    gmean = tf.sqrt(syhy*(yl-tf.reduce_sum(y_pred)-sy+syhy)/(sy*(yl-sy)))
    return 1-gmean

################################ Ours_BAccu ################################
def Ours_BAccu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    syhy = tf.reduce_sum(y_true*y_pred)
    sy = tf.reduce_sum(y_true)
    yl = y_train.shape[0]
    baccu = (yl*(syhy+sy)-sy*(tf.reduce_sum(y_pred)+sy)) / (2*sy*(yl-sy))
    return 1-baccu

############ Common Functions ############
def get_results(y, predicted):
#     print("Conf. Mat:\n", pd.DataFrame(metrics.confusion_matrix(y, predicted)).rename(index={0:'Real(-1/0)', 1:'Real(1)'}, columns={0:'Pred(-1/0)', 1:'Pred(1)'}))  
    TN = metrics.confusion_matrix(y, predicted)[0,0]
    FP = metrics.confusion_matrix(y, predicted)[0,1]
    FN = metrics.confusion_matrix(y, predicted)[1,0]
    TP = metrics.confusion_matrix(y, predicted)[1,1]
#     print("TN, FP, FN, TP:", TN, FP, FN, TP)
    acc = (TP+TN)/(TP+TN+FP+FN)
#     acc.append(acc)
#     pre = TP/(TP+FP)
#     met.append(pre)
#     rec = TP/(TP+FN)
#     met.append(rec)
    f1 = TP/(TP + 0.5*(FP+FN))
#     f1.append(f1)
    f05 = TP/(TP + 0.8*FP + 0.2*FN)
#     f05.append(f05)
    f2 = TP/(TP + 0.2*FP + 0.8*FN)
#     f2.append(f2)
    gmean = ((TP/(TP+FN)) * (TN/(TN+FP)))**0.5
#     gmean.append(gmean)
    bacc = 0.5*(TP/(TP+FN) + TN/(TN+FP))
#     bacc.append(b_acc)

#     print('Accuracy = {: .4f}'.format(acc))
#     print('Precesion = {: .4f}'.format(pre))
#     print('Recall = {: .4f}'.format(rec))
#     print('F1 = {: .4f}'.format(f1))
#     print('F0.5 = {: .4f}'.format(f05))
#     print('F2 = {: .4f}'.format(f2))
#     print('Gmean = {: .4f}'.format(gmean))
#     print('Balanced Accu. = {: .4f}'.format(bacc))
    
    return np.round(acc,4), np.round(f1,4), np.round(f05,4), np.round(f2,4), np.round(gmean,4), np.round(bacc,4)

## Metadata of 105 Datasets

In [ ]:
data = []
sample = []
feature = []
imbalance1 = []
imbalance2 = []

In [ ]:
# Experiments for 105-3 Datasets
for i in range(1, 106):
    if i == 23 or i == 82 or i == 84:
        continue
    df = pd.read_csv('ds'+ str(i) +'.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    print('<Original Class>\n', df.iloc[:,-1].value_counts())
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0]
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0]
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "{: .2f}:1".format(df.iloc[:,-1].value_counts()[0]/df.iloc[:,-1].value_counts()[1]))
    data.append(i)
    imbalance1.append(df.iloc[:,-1].value_counts()[0]/len(df))
    imbalance2.append(df.iloc[:,-1].value_counts()[1]/len(df))
    
    X = df.iloc[:, :-1]
    X = (X - X.mean())/X.std()    # Features // Standardization
    y = df.iloc[:, -1]
    sample.append(X.shape[0])
    feature.append(X.shape[1])

In [ ]:
print(data)

In [ ]:
print(sample)

In [ ]:
print(feature)

In [ ]:
print(imbalance1)

In [ ]:
print(imbalance2)

In [ ]:
pd.options.display.max_rows = 110

In [ ]:
df_im = pd.DataFrame(data, columns=["data#"] )
# df_im = pd.DataFrame(dummy, columns=["&1"] )
df_im["sample"] = sample
# df_im["&2"] = "&"
df_im["feature"] = feature
# df_im["&3"] = "&"
df_im["negative(0)"] = imbalance1
# df_im[":"] = ":"
df_im["positive(1)"] = imbalance2
# df_im["\\"] = "\\"+"\\"
df_im

In [ ]:
list_90 = list(df_im[df_im["negative(0)"] > 0.9]['data#'])
list_80 = list(df_im[(df_im["negative(0)"] > 0.8) & (df_im["negative(0)"] <= 0.9)]['data#'])
list_70 = list(df_im[(df_im["negative(0)"] > 0.7) & (df_im["negative(0)"] <= 0.8)]['data#'])
list_60 = list(df_im[(df_im["negative(0)"] > 0.6) & (df_im["negative(0)"] <= 0.7)]['data#'])
print(list_90, len(list_90))
print(list_80, len(list_80))
print(list_70, len(list_70))
print(list_60, len(list_60))

## Experiments of 105 Datasets

In [ ]:
from keras import initializers
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=100
beta = 1
learning_rate=0.001         # This is changeable (0.001/0.01/0.1)

In [ ]:
# Experiments for 105-3 Datasets

for i in range(1, 106):
    
    if i == 23 or i == 82 or i == 84:
        continue
    res = pd.DataFrame([i for x in range(100)], columns=['Dataset#'])
    res['L'] = 0
    df = pd.read_csv('ds'+ str(i) +'.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    print('<Original Class>\n', df.iloc[:,-1].value_counts())
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0]
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0]
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "{: .2f}:1".format(df.iloc[:,-1].value_counts()[0]/df.iloc[:,-1].value_counts()[1]))
    
    X = df.iloc[:, :-1]
    X = (X - X.mean())/X.std()    # Features // Standardization
    y = df.iloc[:, -1]
    
    for t in range(1):   # 4 times repeat
        print('#'*50,'attempt={0}'.format(t+1),'#'*50)
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)
        fscore = []
                
        for j in range(100): # 100 of L
            if t == 0: 
                res.iloc[j, 1] = 3*j + 1
                
            print('!'*50,'L={0}'.format(3*j+1),'!'*50)
            L = 3*j + 1
            f1_list = []
            
            n_iter=0
            for train_index, test_index in skf.split(df, df.iloc[:,-1]):
                n_iter += 1
                X_train = X.iloc[train_index]
                y_train= y.iloc[train_index]
                X_test = X.iloc[test_index]
                y_test= y.iloc[test_index]
#                 print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
                X_train = np.array(X_train)
                y_train = np.array(y_train)
                y_train = y_train.astype(float)
                X_test = np.array(X_test)
                y_test = np.array(y_test)
                y_test = y_test.astype(float)
                
                batch_size = int(X_train.shape[0] * 0.05)
                #########################################################################################################
                model = Sequential()
                model.add(Dense(hidden_node, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
                model.add(BatchNormalization())
                model.add(Activation(activation))
                model.add(Dense(1, activation='sigmoid'))
                opt = optimizers.Adam(learning_rate = learning_rate)
    
                model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
                history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)   
                result = 1/(1+tf.math.exp(-L*(model.predict(X_test)-0.5)))
                if np.all(np.isnan(result)):
                    f1_list.append(0)
                else:
                    predicted = np.round(result)
                    acc, f1, f05, f2, gmean, bacc = get_results(y_test, predicted)
                    f1_list.append(f1)
#             plt.plot(history.history['loss'], label='loss')
#             plt.ylim([0, 1])
#             plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#             plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#             plt.title("Cost Function",fontweight="bold",fontsize = 20)
#             plt.legend()
#             plt.show()
            print('F1 =', np.mean(f1_list), f1_list)
            fscore.append(np.mean(f1_list))
        res['att_{}'.format(t+1)] = fscore    
    print(res)
    res.to_csv("5CV_MLP_102_L1to4_001.csv", mode = 'a', float_format='%.4g')